# Solar Panel Design

This notebook explores a first-pass PV design model for Utrecht. It accepts multiple roof segments with orientation, tilt, and installed capacity, then shows hourly, monthly, and per-segment output with Plotly.

In [18]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from home_energy.domain import PVSegment
from home_energy.solar import (
    UTRECHT_SITE,
    build_hourly_figure,
    build_monthly_figure,
    build_segment_comparison_figure,
    compass_label_for_azimuth,
    simulate_pv_system,
)

In [19]:
segments = [
    PVSegment(name='South roof', capacity_kwp=5.0, azimuth_deg=180.0, tilt_deg=35.0),
    PVSegment(name='East roof', capacity_kwp=3.00, azimuth_deg=90.0, tilt_deg=25.0),
    PVSegment(name='West roof', capacity_kwp=0.0025, azimuth_deg=270.0, tilt_deg=25.0),
]

temperature_loss_percent = 0.04
compass_label_for_azimuth(segments[0].azimuth_deg)

'South'

In [20]:
result = simulate_pv_system(
    segments,
    year=2026,
    site=UTRECHT_SITE,
    temperature_loss_percent=temperature_loss_percent,
)

for row in result.summary_rows():
    print(row)

{'segment': 'South roof', 'capacity_kwp': 5.0, 'azimuth_deg': 180.0, 'tilt_deg': 35.0, 'annual_kwh': 7878.459326609025}
{'segment': 'East roof', 'capacity_kwp': 3.0, 'azimuth_deg': 90.0, 'tilt_deg': 25.0, 'annual_kwh': 3480.4022632393257}
{'segment': 'West roof', 'capacity_kwp': 0.0025, 'azimuth_deg': 270.0, 'tilt_deg': 25.0, 'annual_kwh': 2.9006370741568106}
{'segment': 'Total', 'capacity_kwp': 8.0025, 'azimuth_deg': nan, 'tilt_deg': nan, 'annual_kwh': 11361.76222692252}


In [21]:
hourly_figure = build_hourly_figure(result)
hourly_figure

In [22]:
monthly_figure = build_monthly_figure(result)
monthly_figure

In [23]:
comparison_figure = build_segment_comparison_figure(result)
comparison_figure

## Summer vs Winter Day Snapshot

The yearly simulation includes all seasons. Below we compare one representative summer day and one representative winter day in Utrecht:

- Summer: June 21
- Winter: December 21

This helps show the seasonal change in daylight window and total daily production for the same PV setup.

In [24]:
import plotly.graph_objects as go


def _daily_curve_from_result(sim_result, month: int, day: int):
    hourly = [0.0] * 24
    for ts, value in zip(sim_result.timestamps, sim_result.total_hourly_kwh):
        if ts.month == month and ts.day == day:
            hourly[ts.hour] = value
    return hourly


summer_label = "Jun 21"
winter_label = "Dec 21"

summer_curve = _daily_curve_from_result(result, 6, 21)
winter_curve = _daily_curve_from_result(result, 12, 21)

summer_total = sum(summer_curve)
winter_total = sum(winter_curve)

seasonal_day_figure = go.Figure()
seasonal_day_figure.add_trace(
    go.Scatter(x=list(range(24)), y=summer_curve, mode="lines", name=f"{summer_label} ({summer_total:.1f} kWh)")
)
seasonal_day_figure.add_trace(
    go.Scatter(x=list(range(24)), y=winter_curve, mode="lines", name=f"{winter_label} ({winter_total:.1f} kWh)")
)
seasonal_day_figure.update_layout(
    title="Daily PV Output: Summer vs Winter",
    xaxis_title="Hour of day",
    yaxis_title="Energy per hour (kWh)",
)
seasonal_day_figure